In [1]:
!pip install torch==2.2.0
!pip install portalocker==2.8.2
!pip install numpy==1.26.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("train_2.csv")

In [3]:
df.head(5)

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,...,"Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.08,6.03,147.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,75147744.85
1,0,2023,1,1.32,6.04,162.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,74914754.22
2,0,2025,1,0.82,6.00,145.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,87125506.92
3,0,2025,2,0.90,6.00,118.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,82659801.63
4,0,2024,2,1.25,6.06,154.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,74209339.11


In [4]:
df["period"] = df["Год"].astype(str) + "-" + df["Месяц"].astype(str).str.zfill(2)
df = df.sort_values(["new_id", "Год", "Месяц"]).reset_index(drop=True)

In [ ]:
df

In [5]:
target = "РТО"
id = "new_id"

In [6]:
group = df.groupby(id)

Лаги - временные окна в месяцах

In [7]:
for lag in [1, 2, 3, 12, 13]:
    df[f"rto_lag_{lag}"] = group[target].shift(lag)

Скользящее среднее

In [8]:
for window in [3, 6, 12]:
    df[f"rto_roll_mean_{window}"] = group[target].transform(
        lambda s: s.shift(1).rolling(window).mean()
    )

In [9]:
train_df = df[df["period"] <= "2024-11"].copy()

In [10]:
valid_df = df[df["period"].isin(["2024-12", "2025-01"])].copy()

In [11]:
test_df = df[df["period"] == "2025-02"].copy()

In [12]:
# Удаляем строки без лагов

lag_features = [
    "rto_lag_1",
    "rto_lag_2",
    "rto_lag_3",
    "rto_lag_12",
    "rto_lag_13",
    "rto_roll_mean_3",
    "rto_roll_mean_6",
    "rto_roll_mean_12",
]

train_df = train_df.dropna(subset=lag_features + [target])
valid_df = valid_df.dropna(subset=lag_features + [target])
test_df = test_df.dropna(subset=lag_features + [target])

In [13]:
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test:", test_df.shape)

train: (125290, 33)
valid: (25058, 33)
test: (12529, 33)


In [14]:
# Выбираем признаки для нейросети

num_features = [
    "rto_lag_1",
    "rto_lag_2",
    "rto_lag_3",
    "rto_lag_12",
    "rto_lag_13",
    "rto_roll_mean_3",
    "rto_roll_mean_6",
    "rto_roll_mean_12",
    "month_sin",
    "month_cos",
    # дополнительные числовые признаки магазина
    "Рабочие часы в день",
    "Численность населения",
    "Количество домохозяйств",
    "Трафик пеший, в час",
    "Трафик авто, в час",
    "Маркетплейсы, доставки, постаматы (100 м)",
    "Медицинские уч. и аптеки (300 м)",
    "Школы (300 м)",
    "Остановки (300 м)",
    "Продуктовые магазины (500 м)",
    "Пятерочки (500 м)",
    "Количество касс",
    "Флаг алкогольной лицензии",
]

cat_features = [
    "Дата открытия, категориальный",
    "Торговая площадь, категориальный",
    "Населенный пункт",
    "Регион",
]

In [15]:
# One-hot encoding категориальных признаков

train_df["sample_type"] = "train"
valid_df["sample_type"] = "valid"
test_df["sample_type"] = "test"

full_df = pd.concat([train_df, valid_df, test_df], axis=0)

full_df = pd.get_dummies(full_df, columns=cat_features, drop_first=False)

train_df = full_df[full_df["sample_type"] == "train"].copy()
valid_df = full_df[full_df["sample_type"] == "valid"].copy()
test_df = full_df[full_df["sample_type"] == "test"].copy()

train_df = train_df.drop(columns=["sample_type"])
valid_df = valid_df.drop(columns=["sample_type"])
test_df = test_df.drop(columns=["sample_type"])

all_features = [
    col for col in train_df.columns if col not in ["РТО", "period", "new_id"]
]

In [16]:
# Формируем X и y

X_train = train_df[all_features].astype("float32")
X_valid = valid_df[all_features].astype("float32")
X_test = test_df[all_features].astype("float32")

y_train = train_df[target].values.astype("float32")
y_valid = valid_df[target].values.astype("float32")
y_test = test_df[target].values.astype("float32")

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)
print(X_test.shape, y_test.shape)

(125290, 2742) (125290,)
(25058, 2742) (25058,)
(12529, 2742) (12529,)


In [17]:
# Масштабирование признаков

feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0)

# чтобы не делить на 0
feature_std = feature_std.replace(0, 1)

X_train_scaled = ((X_train - feature_mean) / feature_std).values
X_valid_scaled = ((X_valid - feature_mean) / feature_std).values
X_test_scaled = ((X_test - feature_mean) / feature_std).values

In [18]:
# Масштабирование target

y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

target_mean = y_train_log.mean()
target_std = y_train_log.std()

y_train_scaled = (y_train_log - target_mean) / target_std
y_valid_scaled = (y_valid_log - target_mean) / target_std

print("target_mean:", target_mean)
print("target_std:", target_std)

target_mean: 18.244455
target_std: 0.4599085


In [21]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [22]:
# Dataset и DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_valid_tensor = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

train_dataset = TensorDataset(y_train_tensor, X_train_tensor)
valid_dataset = TensorDataset(y_valid_tensor, X_valid_tensor)

BATCH_SIZE = 256

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataloader = DataLoader(X_test_tensor, batch_size=BATCH_SIZE, shuffle=False)

In [23]:
# Полносвязная нейросеть
class RTOFullyConnectedModel(nn.Module):

    def __init__(self, input_dim, hidden_dim=256, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, features):
        output = self.fc1(features)
        output = torch.relu(output)
        output = self.dropout(output)

        output = self.fc2(output)
        output = torch.relu(output)
        output = self.dropout(output)

        output = self.fc3(output)

        return output.squeeze(1)

In [24]:
input_dim = X_train_scaled.shape[1]

model = RTOFullyConnectedModel(input_dim=input_dim, hidden_dim=256, dropout=0.1).to(
    device
)

model

RTOFullyConnectedModel(
  (fc1): Linear(in_features=2742, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [25]:
# Функции для обратного преобразования target и MAPE


def inverse_target(y_scaled):
    """
    Возвращает прогноз из масштаба нейросети обратно в РТО.
    """
    y_log = y_scaled * target_std + target_mean
    y_rto = np.expm1(y_log)
    return y_rto


def mape(y_true, y_pred):
    """
    Mean Absolute Percentage Error.
    """
    return np.mean(np.abs(y_pred - y_true) / y_true) * 100

In [26]:
def train(dataloader, epoch, optimizer, criterion):
    model.train()

    total_loss = 0
    total_count = 0

    start_time = time.time()

    for target, features in dataloader:
        target = target.to(device)
        features = features.to(device)

        optimizer.zero_grad()

        predicted = model(features)
        loss = criterion(predicted, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(target)
        total_count += len(target)

    avg_loss = total_loss / total_count

    print(
        "| epoch {:3d} | train loss {:8.5f} | time {:5.2f}s".format(
            epoch, avg_loss, time.time() - start_time
        )
    )

In [27]:
def evaluate(dataloader, criterion, y_true):
    model.eval()

    total_loss = 0
    total_count = 0

    predictions = []

    with torch.no_grad():
        for target, features in dataloader:
            target = target.to(device)
            features = features.to(device)

            predicted = model(features)
            loss = criterion(predicted, target)

            total_loss += loss.item() * len(target)
            total_count += len(target)

            predictions.append(predicted.cpu().numpy())

    predictions = np.concatenate(predictions)
    predictions_rto = inverse_target(predictions)

    return {
        "loss": total_loss / total_count,
        "mape": mape(y_true, predictions_rto),
        "predictions": predictions_rto,
    }


def predict(dataloader):
    model.eval()

    predictions = []

    with torch.no_grad():
        for features in dataloader:
            features = features.to(device)

            predicted = model(features)
            predictions.append(predicted.cpu().numpy())

    predictions = np.concatenate(predictions)
    predictions_rto = inverse_target(predictions)

    return predictions_rto

In [28]:
EPOCHS = 12
LR = 0.05

criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

valid_mape_history = []

for epoch in range(1, EPOCHS + 1):
    train(train_dataloader, epoch, optimizer, criterion)

    valid_result = evaluate(valid_dataloader, criterion, y_true=y_valid)

    valid_mape_history.append(valid_result["mape"])

    scheduler.step()

    print("-" * 70)
    print(
        "| end of epoch {:3d} | valid loss {:8.5f} | valid MAPE {:8.4f}% | lr {:8.5f}".format(
            epoch,
            valid_result["loss"],
            valid_result["mape"],
            scheduler.get_last_lr()[0],
        )
    )
    print("-" * 70)

| epoch   1 | train loss  0.12931 | time 28.61s
----------------------------------------------------------------------
| end of epoch   1 | valid loss  0.10285 | valid MAPE  11.9412% | lr  0.04500
----------------------------------------------------------------------
| epoch   2 | train loss  0.06823 | time 28.03s
----------------------------------------------------------------------
| end of epoch   2 | valid loss  0.10098 | valid MAPE  12.0853% | lr  0.04050
----------------------------------------------------------------------
| epoch   3 | train loss  0.06080 | time 30.04s
----------------------------------------------------------------------
| end of epoch   3 | valid loss  0.10110 | valid MAPE  12.3098% | lr  0.03645
----------------------------------------------------------------------
| epoch   4 | train loss  0.05695 | time 30.14s
----------------------------------------------------------------------
| end of epoch   4 | valid loss  0.09981 | valid MAPE  12.3572% | lr  0.03281

In [29]:
test_predictions = predict(test_dataloader)

test_mape = mape(y_test, test_predictions)

print(f"Test MAPE на феврале 2025: {test_mape:.4f}%")

Test MAPE на феврале 2025: 13.0238%
